# Tips 01：Excel売上データを自動集計

**業務の困りごと**：Excelで担当者別・商品別の売上を手作業で集計する。

**このTipsでできること**：`read_excel` で表を読み、`groupby` で同じ担当者・商品・月の行をまとめて金額を合計します。`as_index=False` にすると集計結果も通常の列として扱えます。

**使用ライブラリ**：pandas, openpyxl

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

1行が1回の売上を表します。「売上金額」は数値、「日付」は年月日です。

最後の `display()` で練習用データ全体を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
import pandas as pd
from pathlib import Path
base = Path('/content') if Path('/content').exists() else Path.cwd()
# 練習用Excelの保存先を決める
src = base / 'sample_sales.xlsx'
# 売上の明細を表にしてExcelへ保存する
sample = pd.DataFrame({
    '日付': ['2026-09-01', '2026-09-02', '2026-09-03', '2026-09-04'],
    '店舗': ['鹿児島', '霧島', '鹿児島', '霧島'],
    '担当者': ['田中', '佐藤', '佐藤', '田中'],
    '商品': ['茶', '菓子', '茶', '菓子'],
    '売上金額': [1200, 800, 1500, 900],
})
sample.to_excel(src, index=False)
print('練習用ファイル:', src)
print('練習用データ（先頭5行）:')
display(pd.read_excel(src))


## 2. 自動処理

`read_excel` で表を読み、`groupby` で同じ担当者・商品・月の行をまとめて金額を合計します。`as_index=False` にすると集計結果も通常の列として扱えます。

**結果を見るポイント**：担当者別と商品別の合計は、どちらも元データの総売上 **4,400円** と一致します。Excelには明細と3種類の集計シートができます。

保存した成果物を読み直して `display()` で表示します。画面の表とダウンロードしたファイルの内容を照らし合わせてください。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
# 作成したExcelを読み込む
df = pd.read_excel(src)
# 日付を日時型に変換し、月別集計に備える
df['日付'] = pd.to_datetime(df['日付'])
# 担当者ごとに売上金額を足す
by_staff = df.groupby('担当者',as_index=False)['売上金額'].sum()
# 商品ごとに売上金額を足す
by_product = df.groupby('商品',as_index=False)['売上金額'].sum()
# 日付から年月を取り出して合計する
by_month = (
    df.assign(月=df['日付'].dt.strftime('%Y-%m'))
      .groupby('月', as_index=False)['売上金額']
      .sum()
)
out = base/'Tips01_売上集計.xlsx'
# 1つのExcelに複数の集計シートを作る
with pd.ExcelWriter(out) as writer:
    df.to_excel(writer,sheet_name='元データ',index=False)
    by_staff.to_excel(writer,sheet_name='担当者別',index=False)
    by_product.to_excel(writer,sheet_name='商品別',index=False)
    by_month.to_excel(writer,sheet_name='月別',index=False)
print(by_staff, '\n', by_product)

# 保存したExcelを読み直し、各シートを表として確認する
for sheet_name, preview in pd.read_excel(out, sheet_name=None).items():
    print(f'【{sheet_name}】 {len(preview)} 行')
    display(preview)


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

担当者別と商品別の合計は、どちらも元データの総売上 **4,400円** と一致します。Excelには明細と3種類の集計シートができます。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert by_staff['売上金額'].sum() == df['売上金額'].sum() == 4400
assert out.exists()
print('確認OK:', out)


## 4. 持ち帰り・練習

**練習問題**：「店舗別」シートも追加してみましょう。



**実務で使う前に**：Excelの1行を別の取引に変えて、再実行したとき集計値が変わることを確かめましょう。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
